# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hajergafsi/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Rebuilds the same feature-window + target-window dataset used in ML-08, so this notebook can re-run the Week-5 model and audit it directly. `label_declined` is again a scoring target only, never a feature.

In [1]:
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("HF_TOKEN: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

MONTH = "2026-03"
FACT = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"
T = "2026-03-15"

data = con.execute(f"""
    WITH feat AS (
        SELECT content_hash_id, client_hash_id,
               AVG(gsc_impressions)      AS avg_impressions,
               AVG(gsc_clicks)           AS avg_clicks,
               AVG(gsc_avg_position)     AS avg_position,
               AVG(gsc_clicks) / NULLIF(AVG(gsc_impressions), 0) AS ctr,
               COUNT(DISTINCT report_date) AS days_seen
        FROM '{FACT}'
        WHERE report_date < DATE '{T}'
        GROUP BY 1, 2
    ),
    tgt AS (
        SELECT content_hash_id, client_hash_id,
               AVG(gsc_impressions) AS tgt_avg_impressions
        FROM '{FACT}'
        WHERE report_date >= DATE '{T}'
        GROUP BY 1, 2
    )
    SELECT f.*, t.tgt_avg_impressions,
           CASE WHEN f.avg_impressions > 5
                 AND t.tgt_avg_impressions < 0.8 * f.avg_impressions
                THEN 1 ELSE 0 END AS label_declined
    FROM feat f
    JOIN tgt t USING (content_hash_id, client_hash_id)
""").df().dropna(subset=['avg_impressions', 'avg_clicks', 'avg_position', 'ctr', 'days_seen'])

FEATURES = ['avg_impressions', 'avg_clicks', 'avg_position', 'ctr', 'days_seen']
print("Shape:", data.shape, "| base rate:", round(data['label_declined'].mean(), 4))

HF_TOKEN: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (150443, 9) | base rate: 0.1857


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

*(Paper: "The State of AI-Driven SEO", March 2026, FlyRank.)*

---

**Finding 1 — Finding #4, "The Freshness Multiplier" (p.9):** the paper reports a 283:1 growth-to-decline ratio in the `361+` days-since-update bucket, and highlights that 365+ day content refreshed within 30 days shows a 3.2x health boost and 57x more impressions.

- **Where does the label come from?** The growth-to-decline ratio comes from `trend_direction`, an observed 30-day-vs-previous-30-day impression comparison (per the glossary, p.5) — a legitimate outcome-based label, not a hand-tuned flag. Good practice.
- **Does the validation design support the claim as stated?** The paper is careful here too — it explicitly flags that the `361+` bucket's 283:1 ratio comes from "283 growing pages versus only 1 declining" page, and states in its own words that this bucket is "too small and too unstable to treat as a headline multiplier" (p.9). It correctly separates this from the `31-90` day bucket's 7.88:1 ratio, which it treats as the real, stable headline number. This is exactly the base-rate/sample-size discipline my own Section 4 (weak picks) practices — flagging low-`days_seen` rows as noisy rather than trusting them at face value.
- **Is the reported metric next to a base rate?** Yes, effectively — the paper prints the raw counts behind the ratio (283 vs 1), which is exactly what lets a reader catch the instability themselves. That's good practice I want to carry into my own ML-07/ML-08 write-ups: don't just print a rate, print the n behind it.
- **My constructive question, tied to my own lane (Refresh / Content Opportunity Scoring):** The 3.2x health boost / 57x impression boost for refreshed 365+ day content is the paper's most attention-grabbing number, and my baseline's whole premise (flagging stale-but-visible pages) leans on this exact pattern being real. Could the paper share the sample size behind the 71-impression "before" figure specifically? If that baseline number is itself built on a small handful of pages (the way the 283:1 ratio was), the 57x multiplier could be similarly unstable even though the paper treats it as a headline number rather than flagging it the way it flagged the 361+ bucket.

**Finding 2 — ML Appendix, "What Predicts Growth?" (p.28):** a Logistic Regression reports 71% holdout accuracy, with Content Age, Days Since Update, and Days Visible as the strongest coefficients separating growing from declining pages.

- **Where does the label come from?** Per the metrics glossary (p.5), "Trend Direction" (growing/declining) is calculated from the 30-day-vs-previous-30-day impression change. That's a legitimate observed-outcome-style label (not a hand-tuned flag), which is good practice — and notably, this is structurally the closest thing in the paper to my own `label_declined` (a feature-window-to-target-window classification), which makes this finding the most directly comparable one to my own Week-5/6 work.
- **Does the validation design support the claim as stated?** The methodology section (p.36) says "Random Forest (80/20 split), Logistic Regression (80/20 split)" but doesn't specify whether that split is random or grouped by brand. With 57 brands in the portfolio and brand-level patterns likely to repeat across a brand's own pages (shared templates, shared content strategy), a plain random 80/20 split could let the model partly memorize brand-level quirks rather than learn transferable signal — exactly the gap my own Section 2 measures on my data.
- **Is the reported metric next to a base rate?** No — 71% accuracy is reported alone, with no stated growing/declining split in this holdout set. Given the trend-direction glossary rule (">10% growth" counted as up), if a majority of the sampled pages are "up" (plausible given the portfolio's stated +70.6% 30-day impression trend on p.3), 71% accuracy could represent real but modest skill above the naive majority-class guess — or very little skill at all if the label is heavily imbalanced. The reader currently can't tell which.
- **My constructive question:** Could the paper share (a) whether the 80/20 split was grouped by brand, and (b) the base rate of the growth/decline label in the holdout set? Both are one extra printed number each, and together they'd let a reader judge how much of the 71% is generalizable skill versus memorized brand structure or majority-class luck.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model already used a grouped-by-client split (the honest design). So the "before/after" comparison here runs the SAME model and features under a plain random split first ("before" — the naive, easier-to-get-a-good-score design) and then the grouped split again ("after" — what I actually reported in ML-08). The gap between them is the finding the `hunting-leakage-and-validating` skill asks for: how much of the score was real skill versus the model quietly memorizing per-client patterns.

In [3]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(df, score_col, label_col, k):
    return df.sort_values(score_col, ascending=False).head(k)[label_col].mean()

def fit_and_score(train_df, test_df, split_name):
    model = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
    model.fit(train_df[FEATURES], train_df['label_declined'])
    scores = model.predict_proba(test_df[FEATURES])[:, 1]
    out = test_df.copy()
    out['score'] = scores
    return {
        'split': split_name,
        'base_rate': round(out['label_declined'].mean(), 4),
        'roc_auc': round(roc_auc_score(out['label_declined'], out['score']), 4),
        'avg_precision': round(average_precision_score(out['label_declined'], out['score']), 4),
        'precision@50': round(precision_at_k(out, 'score', 'label_declined', 50), 4),
    }

# BEFORE -- naive random split, ignoring which client each page belongs to.
train_rand, test_rand = train_test_split(data, test_size=0.3, random_state=42, stratify=data['label_declined'])
before = fit_and_score(train_rand, test_rand, 'BEFORE: random split (naive)')

# AFTER -- grouped by client, same as ML-08.
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(data, groups=data['client_hash_id']))
train_grp, test_grp = data.iloc[tr_idx], data.iloc[te_idx]
overlap = set(train_grp['client_hash_id']) & set(test_grp['client_hash_id'])
after = fit_and_score(train_grp, test_grp, 'AFTER: grouped-by-client split (honest)')

before_after = pd.DataFrame([before, after])
print("Clients overlapping train/test under the grouped split (must be 0):", len(overlap))
before_after

Clients overlapping train/test under the grouped split (must be 0): 0


,split,base_rate,roc_auc,avg_precision,precision@50
0,BEFORE: random split (naive),0.1857,0.8351,0.4613,0.80
1,AFTER: grouped-by-client split (honest),0.2323,0.7904,0.4421,0.38


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**The attack checklist, applied to my final feature set (`avg_impressions`, `avg_clicks`, `avg_position`, `ctr`, `days_seen`):**

- [x] **Timeline drawn:** all 5 features are aggregated only from `report_date < 2026-03-15`; the label (`label_declined`) is computed only from `report_date >= 2026-03-15`. No overlap.
- [x] **No label-derived or sibling columns in the features:** `tgt_avg_impressions` (the column the label thresholds on) is loaded for label construction only and is never in `FEATURES`. Re-confirmed below by deliberately adding it back and watching the score jump, then removing it again.
- [x] **No product flags / existing-system scores as features:** none of `health_score`, `priority_score`, `action_type`, or any refresh flag were ever loaded in this project — they aren't shipped in this dataset by design.
- [x] **Split grouped by the repeating entity:** confirmed in Section 2 (zero overlapping clients between train and test).
- [x] **Base rate printed next to every metric:** done in Section 2's table (`base_rate` column sits beside every score).
- [x] **Top feature importance sanity-checked:** re-run below; checking whether the top feature is a plausible driver or a suspiciously dominant single column.

In [6]:
# Re-confirm leakage detection on the FINAL model (Random Forest), not just the quick check from ML-04.
honest_rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
honest_rf.fit(train_grp[FEATURES], train_grp['label_declined'])
honest_auc = roc_auc_score(test_grp['label_declined'], honest_rf.predict_proba(test_grp[FEATURES])[:, 1])

leaky_features = FEATURES + ['tgt_avg_impressions']
leaky_rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
leaky_rf.fit(train_grp[leaky_features], train_grp['label_declined'])
leaky_auc = roc_auc_score(test_grp['label_declined'], leaky_rf.predict_proba(test_grp[leaky_features])[:, 1])

print("Honest RF (final feature set), ROC-AUC:", round(honest_auc, 4))
print("Leaky RF (+ tgt_avg_impressions), ROC-AUC:", round(leaky_auc, 4))
print("Jump:", round(leaky_auc - honest_auc, 4))
print("\n-> My test harness correctly detects the injected leak (same pattern proven in ML-04).")
print("   tgt_avg_impressions stays OUT of the final feature set.")

Honest RF (final feature set), ROC-AUC: 0.7904
Leaky RF (+ tgt_avg_impressions), ROC-AUC: 0.9474
Jump: 0.157

-> My test harness correctly detects the injected leak (same pattern proven in ML-04).
   tgt_avg_impressions stays OUT of the final feature set.


In [7]:
# Feature importance sanity check on the HONEST final model.
importance = pd.DataFrame({
    'feature': FEATURES,
    'importance': honest_rf.feature_importances_.round(4)
}).sort_values('importance', ascending=False)
print(importance)
print("\nIs the top feature suspiciously dominant (e.g. >0.8 of total importance)?",
      "If so, investigate before trusting the score -- don't celebrate it.")

           feature  importance
0  avg_impressions      0.7267
3              ctr      0.1164
1       avg_clicks      0.0583
2     avg_position      0.0568
4        days_seen      0.0418

Is the top feature suspiciously dominant (e.g. >0.8 of total importance)? If so, investigate before trusting the score -- don't celebrate it.


**Verdict (fill after running):** honest AUC = `0.7904`, leaky AUC = `0.9474`, jump = `0.157` (this should closely mirror the ML-04 result, since the mechanism is identical). Top feature by importance = `avg_impressions` — is that plausible (e.g. `ctr` or `avg_position` driving decline detection) or suspicious? `<FILL>`. My final feature set passes the attack checklist: `<FILL: yes/no, and if no, what's still open>`.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence so far (paste the actual sentence from an earlier notebook — e.g. ML-08's error interpretation or ML-03's framing):**

> The Random Forest model accurately predicts which pages will decline in impressions.

**Rewritten in safe, decision-support language:**

> The Random Forest model, evaluated under a client-grouped split, provides a directional signal of pages likely to experience impression declines based on the `label_declined` definition. This can support prioritization for review, but it is not a guarantee of future decline, and its performance has only been measured on a proxy label within a specific time window.

**What changed between the two versions, and why it matters:** The original sentence implies certainty and a direct causal link ('accurately predicts'). The rewritten version uses cautious language like 'directional signal' and 'likely to experience,' emphasizing that it's a tool for 'decision-support' rather than a definitive forecast. It also clarifies the evaluation context ('client-grouped split,' 'proxy label,' 'specific time window') and explicitly states what the model does *not* guarantee ('not a guarantee of future decline'). This shift from declarative prediction to nuanced insight is crucial for responsible deployment and understanding model limitations.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.